# PEDE — BGE-M3 Search Server (Colab + Cloudflare Tunnel)

Layanan **pencarian hybrid BGE-M3** untuk backend `nsa` (top-k RAG full-text screening Modul 6).

Server ini **memakai ulang `core/vector_store.py`** PEDE, jadi pencarian memakai logika **hybrid (dense + sparse) dengan RRF fusion** yang **identik** dengan ingest — tidak ada duplikasi logika dan tidak ada risiko ruang vektor "geser". Backend `nsa` cukup mengirim query + filter, lalu menerima chunk siap pakai.

**Endpoint:**
- `POST /search` → pencarian hybrid (dense+sparse, RRF) → balikan daftar chunk `{content, score, metadata}`. **Ini yang dipakai `nsa`.**
- `POST /v1/embeddings` *(opsional)* → embedding **dense** OpenAI-compatible, bila ada pemakai yang hanya butuh vektor.

**Cara pakai:**
1. Menu **Runtime → Change runtime type → Hardware accelerator = GPU (T4)**.
2. Menu **Runtime → Run all**.
3. Tunggu sel mencetak **PUBLIC URL** + baris `.env`. Salin ke file `.env` repo `nsa`.
4. Biarkan tab Colab tetap terbuka (sesi berhenti bila idle ~90 menit / maks ~12 jam; jalankan ulang untuk URL baru).

> Penting: model & metode (FlagEmbedding `BGEM3FlagModel`, dense 1024-d + sparse lexical, `max_length=8192`) datang langsung dari `core/vector_store.py`, jadi vektor query berada di **ruang yang sama** dengan chunk di Qdrant. Jalankan dengan **GPU** agar sparse (hybrid) aktif.

In [ ]:
# 1) Install dependencies (1-3 menit)
# Server SEARCH hanya butuh dependency yang diimpor core/vector_store.py.
# Tidak perlu PaddleOCR/PyMuPDF/Gemini (itu khusus pipeline ingest).
!pip install -q FlagEmbedding qdrant-client python-dotenv langchain-text-splitters httpx fastapi "uvicorn[standard]" pycloudflared requests

In [ ]:
# 2) Clone repo PEDE + muat kredensial Qdrant dari Colab Secrets
# Server /search memakai core/vector_store.py langsung, jadi repo harus tersedia
# dan kredensial Qdrant (tempat chunk hasil ingest disimpan) harus terbaca.
import os
from google.colab import userdata

REPO_URL = "https://github.com/ifcoid/PEDE.git"
WORK_DIR = "/content/PEDE"

if not os.path.exists(WORK_DIR):
    print(f"Cloning repo ke {WORK_DIR} ...")
    !git clone {REPO_URL} {WORK_DIR}
else:
    print("Repo sudah ada. git pull untuk update ...")
    !git -C {WORK_DIR} pull
os.chdir(WORK_DIR)


def _secret(name: str) -> str:
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


# Qdrant: collection yang SAMA dengan hasil ingest (pede_colab.ipynb).
os.environ["QDRANT_URL"] = _secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = _secret("QDRANT_API_KEY")
assert os.environ["QDRANT_URL"] and os.environ["QDRANT_API_KEY"], \
    "Secret QDRANT_URL / QDRANT_API_KEY belum diset di panel 🔑 Secrets!"

print("Repo siap di:", os.getcwd(), "| Qdrant secrets OK")

In [ ]:
# 3) Inisialisasi VectorStore (muat BGE-M3 hybrid + konek Qdrant Cloud)
# Memuat model yang SAMA PERSIS dengan ingest (FlagEmbedding, fp16 di GPU, dense+sparse)
# dan logika search hybrid+RRF dari core/vector_store.py.
from core.vector_store import VectorStore

store = VectorStore()          # load model + connect Qdrant (pakai env dari sel sebelumnya)
store.ensure_collection()      # idempotent: pastikan collection + payload index ada
info = store.get_collection_info()

print(f"✅ Siap. backend={store.backend} | hybrid={store.hybrid} | "
      f"dim={store.vector_size} | chunk di Qdrant: {info['points_count']}")
if not store.hybrid:
    print("⚠️  hybrid=False → sparse nonaktif (FlagEmbedding gagal / tanpa GPU). "
          "Pencarian jatuh ke dense-only. Pastikan Runtime = GPU lalu Run all ulang.")

In [ ]:
# 4) API: /search (hybrid RRF → chunk) + /v1/embeddings (OpenAI-compatible, dense)
import secrets
from typing import Union, List, Optional
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

# Token pelindung endpoint publik. nsa kirim 'Authorization: Bearer <token>'.
# Set EMBED_API_KEY di .env nsa = nilai ini. Kosongkan ("") untuk tanpa auth.
API_KEY = secrets.token_hex(16)

app = FastAPI(title="PEDE BGE-M3 Search Server")


def _check(authorization: str):
    if API_KEY and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="invalid api key")


@app.get("/")
def health():
    return {"status": "ok", "model": "BAAI/bge-m3", "dim": store.vector_size,
            "hybrid": store.hybrid, "backend": store.backend}


# --- Endpoint utama: pencarian hybrid (dense+sparse, RRF) → chunk siap pakai ---
class SearchReq(BaseModel):
    query: str
    n_results: int = 10
    article_ids: Optional[List[str]] = None   # batasi ke artikel tertentu
    section_filter: Optional[str] = None       # batasi ke section header tertentu
    doi_filter: Optional[str] = None


@app.post("/search")
def search(req: SearchReq, authorization: str = Header(default="")):
    _check(authorization)
    # Delegasi penuh ke VectorStore.search: hybrid + RRF + filter, identik dgn PEDE.
    results = store.search(
        query=req.query,
        n_results=req.n_results,
        article_ids=req.article_ids,
        section_filter=req.section_filter,
        doi_filter=req.doi_filter,
    )
    return {
        "object": "list",
        "mode": "hybrid" if store.hybrid else "dense",
        "count": len(results),
        "results": results,   # [{content, score, metadata}, ...]
    }


# --- Opsional: embeddings OpenAI-compatible (DENSE saja; sparse tak muat di format ini) ---
class EmbReq(BaseModel):
    input: Union[str, List[str]]
    model: str = "BAAI/bge-m3"


@app.post("/v1/embeddings")
@app.post("/embeddings")
def embeddings(req: EmbReq, authorization: str = Header(default="")):
    _check(authorization)
    texts = [req.input] if isinstance(req.input, str) else list(req.input)
    # _embed_query: dense query vec (ruang sama dgn chunk), konsisten dgn PEDE.
    vecs = [store._embed_query(t)[0] for t in texts]
    return {
        "object": "list",
        "model": req.model,
        "data": [{"object": "embedding", "index": i, "embedding": v} for i, v in enumerate(vecs)],
        "usage": {"prompt_tokens": 0, "total_tokens": 0},
    }


print("API_KEY:", API_KEY)

In [ ]:
# 5) Jalankan server di thread + buka Cloudflare Tunnel → URL publik
import threading, time, uvicorn

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(4)

from pycloudflared import try_cloudflare
url = try_cloudflare(port=8000).tunnel

print("\n" + "=" * 64)
print("PUBLIC URL :", url)
print("=" * 64)
print("\nSalin ke .env repo nsa:\n")
print(f"SEARCH_ENDPOINT={url}/search")
print(f"EMBED_API_KEY={API_KEY}")
print("EMBED_MODEL=BAAI/bge-m3")
print(f"# opsional (hanya jika nsa masih butuh vektor dense): EMBED_ENDPOINT={url}/v1")

In [ ]:
# 6) (Opsional) Uji mandiri — /search harus 200 & balikan chunk berperingkat
import requests
r = requests.post("http://127.0.0.1:8000/search",
                  headers={"Authorization": f"Bearer {API_KEY}"},
                  json={"query": "self-regulated learning in online higher education",
                        "n_results": 3})
print("status:", r.status_code)
data = r.json()
print("mode:", data.get("mode"), "| count:", data.get("count"))
for i, hit in enumerate(data.get("results", []), 1):
    md = hit.get("metadata", {})
    print(f"  {i}. score={hit['score']:.4f} | {str(md.get('title', '?'))[:60]}")

## Setelah dapat URL
1. Tempel `SEARCH_ENDPOINT` + `EMBED_API_KEY` ke `.env` repo **nsa**.
2. `nsa` cukup kirim `POST /search` dengan body `{ "query": "...", "n_results": 10 }` (opsional `article_ids`, `section_filter`, `doi_filter`) dan menerima chunk hasil **hybrid RRF** — logika pencarian identik dengan PEDE, tanpa duplikasi.
3. Selesai dipakai, hentikan sesi Colab (tunnel & GPU bebas). URL kedaluwarsa; jalankan ulang notebook untuk URL baru.

## Penjaga sesi & monitor kesehatan server (jalankan terakhir)
Sel ini menjaga kernel tetap aktif **dan** memverifikasi server embedding benar-benar merespons (bukan sekadar kernel hidup) — menampilkan status, uptime, RAM, GPU, dan PUBLIC URL setiap beberapa menit.

> ⚠️ **Catatan jujur:** ini mengurangi sesi mati karena kernel idle, **tetapi tidak** mencegah Colab memutus sesi akibat tab browser ditutup/tidak aktif (~90 menit) atau batas maksimum ~12 jam. Tetap biarkan tab Colab terbuka. Isi `TG_TOKEN`/`TG_CHAT_ID` (opsional) untuk dapat peringatan Telegram saat server mati/pulih.

In [ ]:
# 7) Penjaga sesi + monitor kesehatan server (biarkan sel ini berjalan)
# PENTING: setelah mengubah TG_TOKEN/TG_CHAT_ID, tekan ■ (stop) lalu Run sel ini lagi.
import time, requests
from datetime import datetime, timezone, timedelta

try:
    import psutil
except ImportError:
    psutil = None

# (Opsional) notifikasi Telegram. Isi KEDUANYA untuk mengaktifkan.
TG_TOKEN = ""       # mis. "123456:ABC-DEF..."
TG_CHAT_ID = ""     # mis. "123456789"

HEALTH_URL = "http://127.0.0.1:8000/"   # endpoint health server lokal
INTERVAL = 300                          # detik antar pengecekan (5 menit)
HEARTBEAT_EVERY = 12                    # kirim "masih hidup" tiap N cek (12 x 5mnt = 1 jam); 0 = nonaktif
WIB = timezone(timedelta(hours=7))      # zona waktu tampilan


def _gpu_stats() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            return f"GPU {(total - free) / 1024**3:.1f}/{total / 1024**3:.1f} GB"
    except Exception:
        pass
    return "GPU n/a"


def _tg(msg: str) -> bool:
    """Kirim pesan Telegram. Return True jika benar-benar terkirim (HTTP 200)."""
    if not (TG_TOKEN and TG_CHAT_ID):
        return False
    try:
        r = requests.post(
            f"https://api.telegram.org/bot{TG_TOKEN}/sendMessage",
            data={"chat_id": TG_CHAT_ID, "text": msg},
            timeout=15,
        )
        if r.status_code != 200:
            # Surface kesalahan token/chat_id (mis. 400 chat not found, 401 unauthorized)
            print(f"  ⚠️ Telegram gagal: HTTP {r.status_code} - {r.text[:200]}")
            return False
        return True
    except Exception as e:
        print("  ⚠️ Telegram error:", e)
        return False


public_url = globals().get("url", "(jalankan sel 4 untuk PUBLIC URL)")
print("🛡️  Monitor server BGE-M3 berjalan. Biarkan sel ini aktif & tab tetap terbuka.")
print(f"    PUBLIC URL : {public_url}")
print(f"    Cek tiap {INTERVAL // 60} menit. Tekan tombol ■ (stop) untuk berhenti.")

# Pesan uji saat mulai -> langsung ketahuan Telegram jalan / salah konfigurasi
if TG_TOKEN and TG_CHAT_ID:
    if _tg(f"🛡️ Monitor PEDE embedding server AKTIF.\nURL: {public_url}"):
        print("    Telegram   : ✅ pesan uji terkirim.")
    else:
        print("    Telegram   : ❌ gagal — cek token/chat_id, dan pastikan Anda sudah /start bot-nya.")
else:
    print("    Telegram   : nonaktif (TG_TOKEN/TG_CHAT_ID kosong).")
print()

start = time.time()
was_down = False
tick = 0
try:
    while True:
        now = datetime.now(WIB).strftime("%H:%M:%S")
        uptime = timedelta(seconds=int(time.time() - start))
        ram = f"RAM {psutil.virtual_memory().percent:.0f}%" if psutil else "RAM n/a"

        # Verifikasi server benar-benar merespons (bukan hanya kernel hidup)
        try:
            ok = requests.get(HEALTH_URL, timeout=10).status_code == 200
        except Exception:
            ok = False

        status = "🟢 server OK" if ok else "🔴 SERVER DOWN"
        print(f"[{now} WIB] {status} | uptime {uptime} | {ram} | {_gpu_stats()}")

        tick += 1
        if not ok and not was_down:
            _tg(f"🔴 PEDE embedding server DOWN ({now} WIB). Periksa Colab.")
            was_down = True
        elif ok and was_down:
            _tg(f"🟢 PEDE embedding server pulih ({now} WIB).")
            was_down = False
        elif ok and HEARTBEAT_EVERY and tick % HEARTBEAT_EVERY == 0:
            # Heartbeat berkala saat sehat -> ada kabar walau status tak berubah
            _tg(f"🟢 PEDE server masih hidup ({now} WIB) | uptime {uptime} | {_gpu_stats()}")

        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\n⏹️ Monitor dihentikan.")